# සිංහල / General AI Chatbot — Colab Runner

මේ notebook එකෙන් GitHub repo එක clone කරලා, QLoRA train කරලා, Gradio chat UI එක ක්‍රියාත්මක කරන්න පුළුවන්.

**මුලට:** `Runtime` → `Change runtime type` → `GPU` (T4) තෝරන්න. ඊට පස්සේ සියලු cells එකපස්සේ run කරන්න.

In [ ]:
# 1) Packages install කරන්න (Unsloth = වේගවත් 4-bit training)
!pip install -q unsloth "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    trl peft accelerate bitsandbytes datasets transformers gradio openai huggingface_hub

print('✅ packages installed')

In [ ]:
# 2) Repo clone කරලා ඇතුලට යන්න
!git clone https://github.com/hirushanethsara358-bot/qlora-sinhala-task.git
%cd qlora-sinhala-task

!ls

## 3) Basic Train (QLoRA SFT) — demo

මේක demo පමණයි (`--max_steps 30`). හොඳටම train කරන්න නම් cell 3b (advanced) යොදාගන්න.

In [ ]:
!python train.py --model unsloth/Qwen2.5-7B-Instruct-bnb-4bit --max_steps 30

## 3b) PROPER advanced training (SFT → ORPO)

වැඩි data (SFT 4000 + preferences 3000), උස් LoRA rank (64), context 4096, පුද්ගලික steps.
Free T4 එකේ විනාඩි 30–90 යනුම්. Session බිඳුණොත් cell 4 මාර්ගයෙන් HF Hub එකට save කරගන්න.

In [ ]:
# Proper training (defaults already set for quality)
!python train_advanced.py \
    --sft_data "HuggingFaceH4/ultrachat_200k" \
    --pref_data "mlabonne/orpo-dpo-mix-40k" \
    --r 64 --max_seq_length 4096 --sft_steps 400 --orpo_steps 300

## 4) Trained model එක Hugging Face Hub එකට save කරන්න (IMPORTANT)

Colab session එක බිඳුණොත් `outputs/` folder එක නැතිවේ. ඒ නිසා HF Hub එකට upload කරගන්න.
හෙලාට HF account එකේ token එක (`hf_...`) දෙන්න. Repo එක ස්වයංක්‍රීයව හදෙනවා.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxxx'   # <-- ඔයාගේ Hugging Face token එක දන්න
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(folder_path='outputs',
                   repo_id='YOUR_USERNAME/sinhala-general-chatbot',  # <-- username දන්න
                   token=os.environ['HF_TOKEN'])
print('✅ pushed to Hugging Face Hub')

## 5) Chat (Gradio public link)

Train කලේ නම් `--lora outputs` යොදාගන්නවා; නැත්නම් base model එක.

In [ ]:
!python chat.py --lora outputs 2>/dev/null || python chat.py